## Rent prediction model (RQ1)

Trains a gradient-boosted regression model to predict monthly rent from property
characteristics, benchmarked against a linear regression baseline.

Spatial location is represented as continuous coordinates rather than
categorical district labels, so that the model can generalise across districts
with few observations. Of 205 districts, the median contains four listings.

Inputs:
  data/processed/listings_clean.csv    2,400 London listings
  data/processed/district_spatial.csv  district centroids and distance to centre

Reported: MAE, RMSE, R2 and MAPE, against a linear regression baseline.

In [2]:
import pandas as pd
import numpy as np

listings = pd.read_csv('../data/processed/listings_clean.csv')
spatial = pd.read_csv('../data/processed/district_spatial.csv')

print("Listings:", listings.shape)
print("Spatial: ", spatial.shape)

Listings: (2400, 10)
Spatial:  (329, 6)


In [3]:
df = listings.merge(spatial,
                    left_on='subdistrict_code',
                    right_on='district',
                    how='left')

print("After join:", df.shape)
print("Listings with no spatial match:", df['easting'].isna().sum())

After join: (2400, 16)
Listings with no spatial match: 0


### Feature preparation

Location is represented by easting, northing and distance from central London
rather than by district label, following the approach agreed with the supervisor
to enable generalisation across sparse districts. Categorical property
attributes are one-hot encoded. The target is log-transformed rent; predictions
are converted back to pounds for reporting.

In [4]:
# Features the model will use
numeric = ['BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station',
           'easting', 'northing', 'dist_centre_km', 'is_shared']

categorical = ['PROPERTY TYPE', 'Furnish Type']

# One-hot encode the categorical columns
X = pd.get_dummies(df[numeric + categorical],
                   columns=categorical,
                   drop_first=True)

y = df['rent_log']

print("Features:", X.shape)
print(X.columns.tolist())

Features: (2400, 32)
['BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station', 'easting', 'northing', 'dist_centre_km', 'is_shared', 'PROPERTY TYPE_Bungalow', 'PROPERTY TYPE_Chalet', 'PROPERTY TYPE_Cottage', 'PROPERTY TYPE_Detached', 'PROPERTY TYPE_Duplex', 'PROPERTY TYPE_End of Terrace', 'PROPERTY TYPE_Flat', 'PROPERTY TYPE_Flat Share', 'PROPERTY TYPE_Ground Flat', 'PROPERTY TYPE_Ground Maisonette', 'PROPERTY TYPE_House', 'PROPERTY TYPE_House Share', 'PROPERTY TYPE_House of Multiple Occupation', 'PROPERTY TYPE_Maisonette', 'PROPERTY TYPE_Mews', 'PROPERTY TYPE_Mobile Home', 'PROPERTY TYPE_Penthouse', 'PROPERTY TYPE_Semi-Detached', 'PROPERTY TYPE_Studio', 'PROPERTY TYPE_Terraced', 'PROPERTY TYPE_Town House', 'Furnish Type_Furnished', 'Furnish Type_Furnished or unfurnished, landlord is flexible', 'Furnish Type_Part furnished', 'Furnish Type_Unfurnished']


### Train and test split

Twenty per cent of listings are held out for evaluation. The random seed is
fixed so that the split is reproducible across runs, allowing the ablation
comparisons in Section 4 to be made on identical data.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)

Train: (1920, 32)
Test:  (480, 32)


### Handling missing bathroom counts

Bathroom counts were set to missing for shared lettings, where the field
describes the whole property rather than the room being let. XGBoost handles
missing values natively by learning a default branch direction. Linear
regression cannot, so for the baseline only these values are imputed with the
training-set median. The imputation uses the training median rather than the
full dataset to avoid leaking test-set information.

In [9]:
# Linear regression cannot handle missing values; XGBoost can.
# For the baseline only, missing bathroom counts are imputed with the median.
X_train_lin = X_train.fillna(X_train['BATHROOMS'].median())
X_test_lin = X_test.fillna(X_train['BATHROOMS'].median())

print("Missing in baseline training set:", X_train_lin.isna().sum().sum())

Missing in baseline training set: 0


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = LinearRegression()
baseline.fit(X_train_lin, y_train)
pred_log = baseline.predict(X_test_lin)

# Convert predictions and actuals from log scale back to pounds
pred_gbp = np.exp(pred_log)
actual_gbp = np.exp(y_test)

print("Linear regression baseline")
print("  R2:   ", round(r2_score(y_test, pred_log), 4))
print("  MAE:  £", round(mean_absolute_error(actual_gbp, pred_gbp)))
print("  RMSE: £", round(np.sqrt(mean_squared_error(actual_gbp, pred_gbp))))
print("  MAPE: ", round(np.mean(np.abs((actual_gbp - pred_gbp) / actual_gbp)) * 100, 1), "%")

Linear regression baseline
  R2:    0.7921
  MAE:  £ 560
  RMSE: £ 1284
  MAPE:  19.9 %


In [11]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

model.fit(X_train, y_train)          # note: X_train, not X_train_lin
pred_log = model.predict(X_test)

pred_gbp = np.exp(pred_log)

print("XGBoost")
print("  R2:   ", round(r2_score(y_test, pred_log), 4))
print("  MAE:  £", round(mean_absolute_error(actual_gbp, pred_gbp)))
print("  RMSE: £", round(np.sqrt(mean_squared_error(actual_gbp, pred_gbp))))
print("  MAPE: ", round(np.mean(np.abs((actual_gbp - pred_gbp) / actual_gbp)) * 100, 1), "%")

XGBoost
  R2:    0.8225
  MAE:  £ 538
  RMSE: £ 1164
  MAPE:  18.3 %


In [12]:
results = pd.DataFrame([
    {'model': 'Linear regression', 'r2': 0.7921, 'mae': 560, 'rmse': 1284, 'mape': 19.9},
    {'model': 'XGBoost', 'r2': 0.8225, 'mae': 538, 'rmse': 1164, 'mape': 18.3},
])
results.to_csv('../results/rq1_baseline.csv', index=False)
print(results)

               model      r2  mae  rmse  mape
0  Linear regression  0.7921  560  1284  19.9
1            XGBoost  0.8225  538  1164  18.3
